In [ ]:
from impala.dbapi import connect
import pandas as pd
import geopandas as gpd
from shapely import wkt
import warnings
warnings.filterwarnings('ignore')

conn = connect(host="172.20.192.154", port=10000, user="datawave", database="purchase_data", auth_mechanism="PLAIN")
# print(conn)
data_path = r"D:\\02_사업관리\\2026년\\데이터 분석 컨설팅\\20260729_중소상공인지원과\\"
output_path = r"D:\\02_사업관리\\2026년\\데이터 분석 컨설팅\\20260729_중소상공인지원과\\집계결과\\"

g100mrkt = pd.read_csv(data_path + "golmokmarket_25ea.csv", sep="|", encoding="utf-8", dtype={'saup_yr': str})
g100mrkt = gpd.GeoDataFrame(g100mrkt, geometry=wkt.loads(g100mrkt['geometry']), crs=5179)
g50m = pd.read_sql("SELECT * FROM purchase_data.l0csv_busan_50cell_v21", conn)
g50m.columns = [col.split('.')[1] for col in g50m.columns]
g50m.drop(['dat_flow_id','part_batchdate'], axis=1, inplace=True)
g50m = gpd.GeoDataFrame(g50m, geometry=wkt.loads(g50m['geometry']), crs=5179)
# g50m = gpd.read_file(r"D:\\02_사업관리\\자체분석\\04_공간DB\\griddata\\bigdata_50m_busan.shp", crs=5179)
mapping= gpd.sjoin(g50m, g100mrkt, how="inner", predicate="intersects")
conn.close()

# 2021년~2025년 성연령 생활인구를 골목상권 25개 영역기준으로 추출
def extract(conn):
    months = [f"{y}{m:02d}" for y in (2021, 2026) for m in range(1, 13)]
    excluded = ('26170','26220','26440','26710')
    for ym in months:
        sql = f"""
        SELECT *
        FROM purchase_data.l0csv_kt_d_lifepop_sex_age_p
        WHERE substr(std_ymd,1,6) = '{ym}' AND substr(hcode,1,5) NOT IN {excluded}
        """
        df = pd.read_sql(sql, conn)
        df.columns = [col.split('.')[1] for col in df.columns]
        df.drop(['dat_flow_id','part_batchdate'], axis=1, inplace=True)
        cols = [col for col in df.columns if 'h_' in col or 'w_' in col or 'v_' in col]
        df[cols] = df[cols].astype(float)
        df['year'] = df['std_ymd'].astype('datetime64[ns]').astype(str).str[:4]
        df = pd.merge(mapping, df, on="id", how="inner")
    return df

# 추출된 데이터프레임을 요청양식 기준으로 성별 생활인구 집계
def aggregate1_emd(conn):
    values = []
    print("2021년~2025년 상권별 성별 생활인구 집계시작")
    df = extract(conn)
    values.append(df)
    concated = pd.concat(values, ignore_index=True)
    gen_dict = {
        'm_sum': [col for col in df.columns if '_m_' in col],
        'w_sum': [col for col in df.columns if '_w_' in col]}
    for key, value in gen_dict.items():
        concated[key] = concated.loc[:, value].sum(axis=1) / 24
    keys = [key for key in gen_dict.keys()]
    group = concated.groupby(['saup_yr','상권명','year'], 
                             as_index=False)[keys].sum().round().sort_values(['saup_yr','year'], ascending=True)
    group.columns = ['사업연도','상권명','기준연도','남성','여성']
    print("2021년~2025년 상권별 성별 생활인구 집계완료")
    conn.close()
    return group

def aggregate2_emd(conn):
    values = []
    print("2021년~2025년 상권별 성별 생활인구 집계시작")
    df = extract(conn)
    values.append(df)
    concated = pd.concat(values, ignore_index=True)
    age_dict = {
        '10le': [col for col in df.columns if col in ('_0009','1014','1519')],
        'w_sum': [col for col in df.columns if '_w_' in col]}
    for key, value in gen_dict.items():
        concated[key] = concated.loc[:, value].sum(axis=1) / 24
    keys = [key for key in gen_dict.keys()]
    group = concated.groupby(['saup_yr','상권명','year'], 
                             as_index=False)[keys].sum().round().sort_values(['saup_yr','year'], ascending=True)
    group.columns = ['사업연도','상권명','기준연도','남성','여성']
    print("2021년~2025년 상권별 성별 생활인구 집계완료")
    conn.close()
    return group

aggregate1_emd(conn).to_excel(output_path + "연도별_상권별_성별_생활인구1.xlsx", engine="openpyxl", index=False)


2021년~2022년 상권별 성별 생활인구 집계시작
2021년~2022년 상권별 성별 생활인구 집계완료


In [ ]:
from impala.dbapi import connect
import pandas as pd
import geopandas as gpd
from shapely import wkt
import warnings
warnings.filterwarnings('ignore')

conn = connect(host="172.20.192.154", port=10000, user="datawave", database="purchase_data", auth_mechanism="PLAIN")

def extract(conn):
    months = [f"{y}{m:02d}" for y in (2021, 2022) for m in range(1, 2)]
    excluded = ('26170','26220','26440','26710')
    for ym in months:
        sql = f"""
        SELECT *
        FROM purchase_data.l0csv_kt_d_lifepop_sex_age_p
        WHERE substr(std_ymd,1,6) = '{ym}' AND substr(hcode,1,5) NOT IN {excluded}
        """
        df = pd.read_sql(sql, conn)
        df.columns = [col.split('.')[1] for col in df.columns]
        df.drop(['dat_flow_id','part_batchdate'], axis=1, inplace=True)
        cols = [col for col in df.columns if 'h_' in col or 'w_' in col or 'v_' in col]
        df[cols] = df[cols].astype(float)
        df['year'] = df['std_ymd'].astype('datetime64[ns]').astype(str).str[:4]
        # df = pd.merge(mapping, df, on="id", how="inner")
    return df

df = extract(conn)

[col for col in df.columns if col in ('_0009', '_1014', '_1519')]


In [3]:
from impala.dbapi import connect
import pandas as pd
import geopandas as gpd
from shapely import wkt
import warnings
warnings.filterwarnings('ignore')

conn = connect(host="172.20.192.154", port=10000, user="datawave", database="purchase_data", auth_mechanism="PLAIN")

grid = pd.read_sql("SELECT * FROM purchase_data.l0csv_busan_50cell_v15", conn)
grid.columns = [col.split('.')[1] for col in grid.columns]
grid.drop(['dat_flow_id','part_batchdate'], axis=1, inplace=True)
grid = gpd.GeoDataFrame(grid, geometry=wkt.loads(grid['geometry']), crs=5179)
grid.to_file(r"D:\\02_사업관리\\자체분석\\04_공간DB\\griddata\\busan_50cell_v15.shp", driver="ESRI Shapefile", encoding="utf-8")